# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook guides you through loading, overviewing, and processing the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library for AI-ready, FAIR-standardized data.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Metadata is accessed as an object
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}")
print(f"Dataset Description: {metadata.description}")
print(f"FAIR Standard Version: {metadata.conformsTo}")
print(f"Date Published: {metadata.datePublished}")
print(f"Keywords: {', '.join(metadata.keywords)}")

## 2. Data Overview
Review available record sets, fields, and their unique `@id`s.

In [ ]:
# List available record sets
record_sets = dataset.record_sets # Each record set is an object
print("Available Record Sets:")
for rs in record_sets:
    print(f"  - Name: {rs.name}")
    print(f"    @id: {rs.id}")
    print(f"    Description: {rs.description}")
    print(f"    Number of fields: {len(rs.fields)}")
    print()

# Show fields for each record set
for rs in record_sets:
    print(f"Fields in Record Set '{rs.name}' (@id: {rs.id}):")
    for field in rs.fields:
        print(f"  - Field Name: {field.name}")
        print(f"    @id: {field.id}")
        print(f"    Data Type: {field.data_type}")
        if field.description:
            print(f"    Description: {field.description}")
        print()
    print("-" * 40)

# Preview records in each record set by id
for rs in record_sets:
    print(f"Sample records from Record Set '{rs.name}' (@id: {rs.id}):")
    records = list(dataset.records(record_set=rs.id))
    for rec in records[:2]:  # show first 2 samples
        print(json.dumps(rec, indent=2))
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s found above.

In [ ]:
# Prepare to load each record set into pandas DataFrames

# List record set @ids
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

# Show columns and preview for the main record set (e.g., the survey responses)
# We'll select the first record set as primary (typically the main survey table)
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"Columns in record set '{main_record_set_id}':")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, and grouping data by key attributes to prepare for analysis.

In [ ]:
# Let's choose example fields for numeric/statistical analysis
# E.g., GAD_7_score (General Anxiety Disorder 7 score), PHQ_9_score (Patient Health Questionnaire 9 score)

record_set_id = main_record_set_id  # Use primary record set
df = dataframes[record_set_id]

# Find a numeric field ID, e.g., 'http://senscience.ai/GAD_7_score' (assuming such an @id structure)
# For demonstration, let's get the exact field name from the columns
numeric_field = None
for col in df.columns:
    if col.lower().startswith('gad_7') or col.lower().startswith('phq_9') or col.lower().startswith('psq'):
        numeric_field = col
        break

if numeric_field:
    print(f"Using field '{numeric_field}' for numeric analysis.")
    # Filter records with score above threshold
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records where {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize the scores in filtered records
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} (z-score):")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a demographic field, e.g. 'level_of_education' or 'gender'
    group_fields = [col for col in df.columns if any(g in col.lower() for g in ["education", "gender", "marital", "village", "age"])]
    group_field = group_fields[0] if group_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Mean {numeric_field} grouped by {group_field}:")
        display(grouped_df.head())
    else:
        print("No suitable grouping field found (e.g. education, gender).")
else:
    print("No suitable numeric field found for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of GAD-7, PHQ-9, or PSQ scores if present
if numeric_field:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field} scores")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Compare score by education/gender if possible
    if group_field:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} Score by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=30)
        plt.show()
else:
    print("No numeric field available to visualize.")

## 6. Conclusion
This exploration demonstrated how to load and process a Croissant schema dataset with `mlcroissant`, referencing all entities by their `@id`s. We:
- Loaded metadata with rich context and provenance
- Identified record sets and fields uniquely using `@id`
- Extracted records and analyzed demographic and psychometric scores
- Visualized score distributions and relationships

**Key findings:** Mental health scores (e.g., GAD-7/PHQ-9) can be filtered and normalized; group analyses by education or gender reveal population differences useful for public health strategy. The FAIR² schema enables precise, reproducible, and transparent AI-ready workflows.

**For further work:** Explore additional record sets, advanced correlations, and integrate domain ontologies via `@id` references for rich AI analysis.